# Next Lab — Free LoRA fine-tuning (Google Colab)

This notebook fine-tunes a small text model using LoRA. It uses a free Colab runtime when available. GPU access is **not guaranteed**, sessions can stop, and this is a notebook workflow—not a persistent production worker.

**Cost guard:** this notebook does not subscribe to or start any paid service. Do not enable paid Colab/Cloud options.

## 1. Select a runtime
In Colab choose **Runtime → Change runtime type → T4 GPU** if offered. If no GPU is available, use CPU only for a tiny smoke test; meaningful fine-tuning may be too slow.

In [ ]:
!pip -q install "transformers>=4.45,<5" "datasets>=3,<5" "peft>=0.13,<1" accelerate bitsandbytes


In [ ]:
import torch, os, json
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('No GPU assigned. Free GPU availability is variable; do not expect long runs on CPU.')


## 2. Upload your dataset
Prepare a `.jsonl` file with one JSON object per line and a `text` field, or a `.csv` file with a `text` column. Export the dataset from your Next Lab account and upload it below. Do not upload secrets or personal data you are not authorized to use.

In [ ]:
from google.colab import files
uploaded = files.upload()
data_path = next(iter(uploaded))
print('Uploaded:', data_path)


In [ ]:
from datasets import load_dataset
if data_path.lower().endswith('.jsonl'):
    ds = load_dataset('json', data_files=data_path, split='train')
elif data_path.lower().endswith('.csv'):
    ds = load_dataset('csv', data_files=data_path, split='train')
else:
    raise ValueError('Use JSONL or CSV with a text column')
if 'text' not in ds.column_names:
    raise ValueError(f'Dataset must contain a text column. Found: {ds.column_names}')
ds = ds.filter(lambda row: isinstance(row['text'], str) and len(row['text'].strip()) > 0)
print(ds)
print('Example:', ds[0]['text'][:300])


## 3. Configure a small base model
Default is `Qwen/Qwen2.5-0.5B-Instruct` to keep the first experiment small. Model downloads may take time and consume Colab disk/bandwidth. Use only models whose license permits your intended use.

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = '/content/next-lab-lora-adapter'
MAX_LENGTH = 512
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model_kwargs = {'torch_dtype': torch.float16 if torch.cuda.is_available() else torch.float32}
if torch.cuda.is_available():
    model_kwargs['device_map'] = 'auto'
    model_kwargs['load_in_4bit'] = True
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj'])
model = get_peft_model(base_model, lora)
model.print_trainable_parameters()


In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4, logging_steps=10, save_strategy='epoch',
    fp16=torch.cuda.is_available(), report_to='none',
    optim='paged_adamw_8bit' if torch.cuda.is_available() else 'adamw_torch',
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized,
                  data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))


## 4. Start training manually
This runs only when you execute this cell. Watch the loss and runtime. If Colab disconnects, the run may stop; save checkpoints frequently for longer experiments.

In [ ]:
train_result = trainer.train()
print(train_result.metrics)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapter saved to', OUTPUT_DIR)


## 5. Download the adapter
The saved output is a LoRA adapter, not a full standalone model. Keep the base model identifier with it.

In [ ]:
!cd /content && zip -qr next-lab-lora-adapter.zip next-lab-lora-adapter
from google.colab import files
files.download('/content/next-lab-lora-adapter.zip')


## What this does not do
- It does not connect to or update your Next Lab database automatically.
- It does not create a persistent GPU worker or production inference endpoint.
- Colab free GPU access, runtime duration, and resource limits vary and are not guaranteed.
- No paid compute is started by this notebook.